<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/logo-bdc.png" align="right" width="64" style="margin-top: 10px;" />

<h1 style="color:#336699; font-weight: normal; margin-top: 0;">Embeddings de Séries Temporais de Imagens de Satélite com sits</h1>
<hr style="border: 2px solid #0077b9; margin-top: 5px; margin-bottom: 20px;" />

<br/>

<div style="text-align: center; font-size: 90%; color: #333;">
    Felipe Carlos<sup><a href="https://orcid.org/0000-0002-3334-4315" target="_blank">
    <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Felipe Souza<sup><a href="https://orcid.org/0000-0002-5826-1700" target="_blank">
    <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Gilberto Camara<sup><a href="https://orcid.org/0000-0002-3681-487X" target="_blank">
        <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Rolf Simoes<sup><a href="https://orcid.org/0000-0003-0953-4132" target="_blank">
        <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    <br/><br/>
    Divisão de Geoinformática e Observação da Terra, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contatos: <a href="mailto:efelipecarlos@gmail.com">efelipecarlos@gmail.com</a>, <a href="mailto:lipecaso@gmail.com">lipecaso@gmail.com</a>
</div>

<br/>

<div style="text-align: justify; margin-left: 15%; margin-right: 15%;">
    <b>Resumo</b><br>
    Este notebook apresenta, de forma introdutória, o uso de <i>embeddings</i> no <code>sits</code>. Discutimos a motivação e a ideia por trás dos <i>embeddings</i>, quando utilizá-los e os passos básicos para gerá-los a partir de séries temporais e usá-los na produção de um mapa de uso e cobertura da terra. Como estudo de caso, utilizamos uma pequena região do estado de Rondônia, na Amazônia brasileira.


</div>    

<br/>

<div style="text-align: justify; margin-left: 15%; margin-right: 15%; font-size: 90%; border: 1px solid #0077b9; border-radius: 6px; padding: 10px;">
    <b>Para um aprofundamento e  discussão completa sobre o <code>sits</code>, consulte o livro em: </b><a href="https://e-sensing.github.io/sitsbook/" target="_blank">SITS: Satellite Image Time Series Analysis on Earth Observation Data Cubes</a>.
</div>

## Ambiente
<hr style="border:1px solid #0077b9;">

Esta seção configura o ambiente de execução deste notebook. As configurações apresentadas não são requisitos do `sits`, mas sim do ambiente utilizado no minicurso.

Este notebook foi pensado para rodar em um computador **sem GPU**: todos os passos foram dimensionados para executar usando apenas a CPU: em um notebook comum, a execução completa leva cerca de 10 a 20 minutos, sem contar o download dos dados.

Instalação do `sits` 2.0.0, a versão descrita no livro:

In [ ]:
devtools::install_github("e-sensing/sits@dev", dependencies = FALSE)

Configuração das dimensões dos plots:

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)

Carregando o `sits`:

In [ ]:
library(sits)

# Usa a CPU para gerar os embeddings do cubo e classificá-lo, mesmo em computadores
# com GPU (por exemplo, Macs com chip Apple).
Sys.setenv(SITS_FORCE_CPU = "TRUE")

Define os diretórios onde os dados baixados e os resultados serão salvos:

In [ ]:
base_dir <- fs::path(Sys.getenv("HOME"))

# Define diretório para os dados de entrada
data_dir <- fs::dir_create(base_dir / "dados/rondonia")

# Define diretório para os dados de saída
output_dir <- fs::dir_create(base_dir / "resultados/embeddings/vicreg-n5000-e30")

## Motivação
<hr style="border:1px solid #0077b9;">

No notebook anterior (*Classificação de Áreas Agrícolas no Oeste da Bahia*), o modelo Random Forest aprendeu diretamente a partir das **bandas originais** das imagens: para cada amostra, ele recebeu os valores de todas as bandas em todas as datas. Essa abordagem funciona muito bem, mas tem um custo: A qualidade do modelo depende diretamente da quantidade e qualidade de amostras **rotuladas** disponíveis. 

Gerar várias amostras com boa qualidade é caro. Exige especialistas, conhecimento da região estudada, trabalho de campo, interpretação visual de imagens e diversos testes.

Por outro lado, imagens de satélite **sem rótulo** existem em abundância. Todo pixel de um cubo de dados tem uma série temporal, e cada uma delas carrega informação sobre como a superfície se comporta ao longo do ano: quando a vegetação cresce, quando o solo fica exposto, quando a área alaga.

A ideia dos *embeddings* é aproveitar essa abundância:

1. **Primeiro**, um modelo de aprendizado profundo (o *encoder*) aprende, a partir de **muitas séries sem rótulo**, a resumir cada série temporal em um vetor curto de números: o *embedding*.
2. **Depois**, um classificador simples aprende a separar as classes de interesse a partir desses resumos, usando **poucas amostras rotuladas**.

Como o *encoder* já aprendeu a "enxergar" os padrões espectro-temporais da região, o classificador tem uma tarefa mais fácil. Além disso, o mesmo *encoder* pode ser reaproveitado em outras tarefas na mesma região.

<div style="text-align: center">
    <video src="videos/09-embeddings-why.mp4" controls preload="metadata" width="85%"></video>
</div>

## O que é um embedding?
<hr style="border:1px solid #0077b9;">

Neste notebook, cada pixel tem uma série temporal com **10 bandas** do Sentinel-2 em **23 datas** ao longo de 2022. São **230 números** por pixel. Muitos desses números são redundantes: bandas vizinhas do espectro variam de forma parecida, e datas próximas costumam ter valores semelhantes.

Um *embedding* é uma forma **compacta** de representar essa série. O *encoder* recebe os 230 números e devolve um vetor com apenas **64 números** (as dimensões do *embedding*, chamadas no `sits` de `EMB01`, `EMB02`, ..., `EMB64`).

O ponto importante é que essa compressão não é arbitrária. Ao final do treinamento, espera-se que:

- séries com **comportamento parecido** (por exemplo, duas áreas de floresta) gerem vetores **próximos**;
- séries com **comportamento diferente** (floresta e área desmatada) gerem vetores **distantes**.

Uma analogia útil: o *embedding* funciona como uma "impressão digital" do comportamento de cada pixel ao longo do ano. Duas impressões parecidas indicam pixels que se comportam de forma parecida, mesmo que nenhum dos dois tenha rótulo.

Uma consequência prática é que as dimensões do *embedding* **não têm significado físico direto**, como as bandas originais (vermelho, infravermelho próximo, etc.). Elas são coordenadas em um espaço aprendido pelo modelo.

<div style="text-align: center">
    <video src="videos/10-what-is-an-embedding.mp4" controls preload="metadata" width="85%"></video>
</div>

## Como o encoder aprende sem rótulos?
<hr style="border:1px solid #0077b9;">

Se não há rótulos, como o *encoder* sabe o que é um "bom" resumo? A resposta é o **aprendizado auto-supervisionado**: o próprio dado fornece a tarefa de treino.

Neste notebook utilizamos o método **VICReg** (*Variance-Invariance-Covariance Regularization*) como base para as explicações e exemplos. Para esse caso, a ideia pode ser entendida em três partes:

1. **Duas versões da mesma série.** A partir de uma série temporal, o `sits` cria duas versões ligeiramente diferentes dela, como se o satélite tivesse passado em datas um pouco diferentes. As duas versões descrevem o mesmo pixel e a mesma trajetória ao longo do ano.
2. **Invariância.** O *encoder* deve produzir *embeddings* **parecidos** para as duas versões. Assim, ele aprende a capturar a **forma** do comportamento sazonal, e não detalhes de uma data específica.
3. **Variância e não redundância.** Ao mesmo tempo, os *embeddings* de pixels diferentes devem continuar **variados**, e cada uma das 64 dimensões deve trazer **informação nova** em relação às demais. Isso impede a "solução preguiçosa" em que o *encoder* devolve o mesmo vetor para tudo.

Ao equilibrar essas exigências, o *encoder* aprende a organizar as séries em um espaço onde comportamentos parecidos ficam próximos, sem nunca ter visto um rótulo.

<div style="text-align: center">
    <video src="videos/11-two-views.mp4" controls preload="metadata" width="85%"></video>
</div>

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; margin: 20px 0; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Outros métodos de aprendizado de representação disponíveis no sits</summary>


Além do VICReg (`sits_ssl_vicreg()`), o `sits` oferece outros métodos para treinar *encoders*. Todos são usados com a mesma função `sits_pre_train()`:

- **Masked Autoencoders** (`sits_ssl_mae()`): esconde parte das datas da série e treina o modelo para reconstruí-las.
- **LeJEPA** (`sits_ssl_lejepa()`): prevê a representação de uma versão da série a partir de outra, mantendo os *embeddings* bem distribuídos.
- **Barlow Twins** (`sits_ssl_barlow_twins()`): faz com que as dimensões dos *embeddings* de duas versões da mesma série concordem entre si, sem serem redundantes.
- **Supervised Barlow Twins** (`sits_barlow_twins()`): como o anterior, mas os pares são formados por amostras diferentes da **mesma classe** (exige rótulos).
- **Supervised Contrastive Learning** (`sits_contrastive_learning()`): aproxima amostras da mesma classe e afasta amostras de classes diferentes (exige rótulos).

</details>

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Para uma descrição detalhada de cada método, consulte o capítulo <a href="https://e-sensing.github.io/sitsbook/emb_methods.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Embedding Methods in SITS</a> do livro oficial do <code>sits</code>.
</div>

## Quando usar embeddings?
<hr style="border:1px solid #0077b9;">

*Embeddings* são especialmente úteis quando:

- **há poucas amostras rotuladas**, mas muitos dados sem rótulo da mesma região;
- a mesma região será mapeada **várias vezes** (por exemplo, variando classes em cada mapeamento), permitindo reaproveitar o *encoder*;
- as classes de interesse dependem de **padrões temporais sutis** (pastagens em diferentes estados, vegetação secundária, transições), que um *encoder* treinado com muitos dados da região pode ajudar a capturar;
- deseja-se construir um **modelo regional** ajustado a uma área e período específicos.

Por outro lado, a classificação direta das bandas (como no notebook anterior) continua sendo uma ótima escolha quando há **amostras rotuladas suficientes** e o problema é pontual: ela é mais simples e mais rápida, pois não exige treinar um *encoder*.

## O fluxo no sits
<hr style="border:1px solid #0077b9;">

O fluxo com *embeddings* acrescenta dois passos ao fluxo de classificação que você já conhece:

| Passo | Função | O que faz |
|---|---|---|
| 1 | `sits_pre_train()` | Treina o *encoder* com amostras **sem rótulo** |
| 2 | `sits_encode()` | Transforma as **amostras rotuladas** e o **cubo** em *embeddings* |
| 3 | `sits_train()` | Treina um classificador com as amostras em *embeddings* |
| 4 | `sits_classify()` | Classifica o cubo de *embeddings* (cubo de probabilidades) |
| 5 | `sits_smooth()` | Suavização espacial das probabilidades |
| 6 | `sits_label_classification()` | Gera o mapa final |

Os passos 3 a 6 são exatamente os mesmos do notebook anterior. A diferença é que, em vez das bandas originais, o classificador recebe os *embeddings*.

<div style="text-align: center">
    <video src="videos/12-embeddings-workflow.mp4" controls preload="metadata" width="85%"></video>
</div>

## Amostras não rotuladas
<hr style="border:1px solid #0077b9;">

Para treinar o *encoder*, precisamos de um conjunto grande de séries temporais que represente bem a região. Utilizamos aqui cerca de 28 mil séries de pontos sorteados aleatoriamente em Rondônia, com imagens Sentinel-2 de 2022 (23 datas, a cada 16 dias). Esse conjunto está disponível no Hugging Face e é carregado com `sits_from_hf()`.

In [ ]:
samples_ssl <- sits_from_hf(
    repo = "e-sensing/samples_book",
    file = "rondonia_30k.parquet",
    type = "dataset"
)

Essas amostras não têm rótulo. Note que todas aparecem como `NoClass`:

In [ ]:
summary(samples_ssl)

Selecionamos apenas as 10 bandas ópticas, deixando de lado os índices espectrais:

In [ ]:
bands <- c("B02", "B03", "B04", "B05", "B06", "B07", "B08", "B11", "B12", "B8A")

samples_ssl <- sits_select(samples_ssl, bands = bands)

Para que o treinamento seja rápido em um computador sem GPU, utilizamos um **subconjunto de 5.000 séries**. Em um caso real, recomenda-se usar o máximo de amostras possível.

In [ ]:
# Define seed para os valores aleatórios gerados abaixo
set.seed(42)

# Cria subconjunto de 5000 amostras aleatórias
samples_ssl <- samples_ssl[sample(nrow(samples_ssl), 5000), ]

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; margin: 20px 0; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Como obter amostras não rotuladas para a sua região</summary>


Amostras como essas podem ser geradas com `sits_random_sampling()`, que sorteia pontos dentro de um cubo, seguida de `sits_get_data()`, que extrai as séries temporais:

```R
random_locations <- sits_random_sampling(
    cube      = cube,
    n_samples = 30000
)

samples_ssl <- sits_get_data(
    cube    = cube,
    samples = random_locations
)
```

Quando existe um mapa de classificação da mesma região, recomenda-se usar `sits_stratified_sampling()`, que equilibra a quantidade de amostras entre classes frequentes e raras.

</details>

## Treinando o encoder
<hr style="border:1px solid #0077b9;">

O *encoder* é treinado com `sits_pre_train()`, que recebe as amostras e o método de aprendizado de representação. Os principais parâmetros do `sits_ssl_vicreg()` são:

- `embedding_dim`: tamanho do *embedding* (aqui, 64 dimensões);
- `encoder_model`: a rede neural que processa as séries (aqui, `sits_tempcnn()`, uma rede convolucional temporal);
- `epochs`: número de passagens pelo conjunto de treino. Usamos um valor reduzido para manter o treino rápido.

In [ ]:
vicreg_encoder <- sits_pre_train(
    samples   = samples_ssl,
    rl_method = sits_ssl_vicreg(
        embedding_dim = 64L,
        encoder_model = sits_tempcnn(),
        epochs        = 30L
    )
)

# Em caso de problemas no treinamento, utilize o encoder pré-treinado:
# vicreg_encoder <- readRDS(url("https://github.com/e-sensing/sitsnotebooks/raw/refs/heads/main/GEOPANTANAL_2026/data/derived/encoder/vicreg-encoder.rds"))

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; margin: 20px 0; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> ⚙️ Mais detalhes sobre o treinamento do encoder</summary>


Os valores padrão do `sits_ssl_vicreg()` (por exemplo, `epochs = 150`) foram pensados para conjuntos grandes e, de preferência, com GPU. Para obter *embeddings* de melhor qualidade em um caso real, considere:

- usar **o máximo** número de amostras possível;
- aumentar o número de épocas (`epochs`);
- testar outros modelos em `encoder_model`, como `sits_lighttae()`.

Consulte a documentação com `?sits_ssl_vicreg` para a lista completa de parâmetros.

</details>

## Gerando embeddings do cubo
<hr style="border:1px solid #0077b9;">

Com o *encoder* treinado, podemos transformar um cubo de dados inteiro em um **cubo de *embeddings***. Usaremos um recorte de cerca de 24 × 24 km do tile `20LMR` (Sentinel-2, 20 m, 2022), também disponível no Hugging Face.

<div style="text-align: center">
    <video src="videos/13-sits-encode.mp4" controls preload="metadata" width="85%"></video>
</div>

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
    <b>Nota:</b> o cubo tem cerca de <b>540 MB</b>. Então, a finalização da operação vai depender da qualidade da conexão que você está utilizando. Os arquivos ficam salvos em <code>data_dir</code> e não são baixados novamente.
</div>

In [ ]:
cube <- sits_from_hf(
    repo       = "e-sensing/book_Rondonia20LMR",
    type       = "dataset",
    output_dir = data_dir
)

cube <- sits_select(cube, bands = bands)

Visualização de uma das datas do cubo, em composição falsa-cor:

In [ ]:
plot(cube, red = "B11", green = "B8A", blue = "B02", dates = "2022-07-16")

Agora, geramos os *embeddings* com `sits_encode()`. Cada pixel do cubo, com suas 10 bandas em 23 datas, é transformado em 64 valores. O resultado é um novo cubo, com 64 "bandas" (`EMB01` a `EMB64`) e uma única data.

**Importante:** as bandas e a quantidade de datas do cubo devem ser as mesmas das amostras usadas para treinar o *encoder*. Neste exemplo, as amostras não rotuladas começam em 2022-01-01 e o cubo em 2022-01-05: as datas estão deslocadas em 4 dias, mas ambos têm 23 datas a cada 16 dias, o que é suficiente para o `sits_encode()`.

In [ ]:
cube_emb <- sits_encode(
    data       = cube,
    encoder    = vicreg_encoder,
    memsize    = 4,
    multicores = 2,
    output_dir = output_dir
)

In [ ]:
sits_bands(cube_emb)

Podemos visualizar três dimensões do *embedding* como uma composição colorida:

In [ ]:
plot(cube_emb, red = "EMB04", green = "EMB02", blue = "EMB03")

As cores não correspondem a bandas físicas, mas regiões com **cores parecidas** têm **comportamento espectro-temporal parecido**. Compare com a composição falsa-cor acima: as áreas de floresta e as áreas abertas continuam distinguíveis, agora representadas pelo *embedding*.

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Para mais detalhes sobre este exemplo, consulte o capítulo <a href="https://e-sensing.github.io/sitsbook/emb_build.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Building regional embeddings: a simple example</a> do livro oficial do <code>sits</code>.
</div>

## Amostras rotuladas no espaço dos embeddings
<hr style="border:1px solid #0077b9;">

Para classificar o cubo, precisamos de amostras **rotuladas**. Utilizamos o conjunto `samples_deforestation_rondonia`, com cerca de 6 mil amostras de 9 classes de uso e cobertura da terra em Rondônia, com as mesmas bandas e datas do cubo.

In [ ]:
samples_labelled <- sits_from_hf(
    repo = "e-sensing/samples_book",
    file = "samples_deforestation_rondonia.parquet",
    type = "dataset"
)

summary(samples_labelled)

Para simular um cenário com menos rótulos, usamos apenas **20%** das amostras:

In [ ]:
# Define seed para os valores aleatórios gerados abaixo
set.seed(42)

# Gera subconjunto com apenas 20% das amostras
samples_labelled <- sits_sample(samples_labelled, frac = 0.2)

# Visão geral das amostras selecionadas
summary(samples_labelled)

As amostras rotuladas também precisam ser transformadas em *embeddings*, com o **mesmo** *encoder* usado no cubo:

In [ ]:
samples_emb <- sits_encode(
    data    = samples_labelled,
    encoder = vicreg_encoder
)

Como visualizar 64 dimensões? O gráfico **t-SNE** projeta os *embeddings* em um plano, preservando as vizinhanças locais: pontos próximos no plano estavam próximos no espaço de 64 dimensões. Já as distâncias entre grupos e o tamanho de cada grupo não têm significado, e o desenho muda um pouco a cada execução.

Se o *encoder* aprendeu bem, amostras da mesma classe tendem a formar **grupos**, mesmo que o *encoder* nunca tenha visto os rótulos.

In [ ]:
plot(samples_emb, mode = "tsne", perplexity = 30)

Observe que água e áreas úmidas, áreas desmatadas e florestas ocupam regiões diferentes do gráfico. Classes com comportamento espectro-temporal parecido, como os três tipos de floresta ou solo exposto e área queimada, se sobrepõem: separá-las é justamente a tarefa do classificador que treinaremos a seguir.

Outra visualização é a de **componentes principais** (PCA), que mostra as duas direções de maior variação dos *embeddings*:

In [ ]:
plot(samples_emb, mode = "PCA")

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; margin: 20px 0; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Mais opções de visualização dos embeddings</summary>


Com `mode = "dimensions"`, o `sits` mostra a distribuição dos valores de cada dimensão do *embedding* para as amostras selecionadas. Por exemplo, para a classe `Wetland`:

```R
# Seleciona apenas amostras de `Wetland`
samples_wet <- sits_select(samples_emb, labels = "Wetland")

# Visualiza amostras com modo `dimensions`
plot(samples_wet, mode = "dimensions")
```

</details>

## Classificação com embeddings
<hr style="border:1px solid #0077b9;">

A partir daqui, o fluxo é o mesmo do notebook anterior. Treinamos um classificador com as amostras em *embeddings*. Utilizamos uma rede **MLP** (*multilayer perceptron*, `sits_mlp()`), adequada para vetores como os *embeddings*.

In [ ]:
mlp_model <- sits_train(
    samples   = samples_emb,
    ml_method = sits_mlp(
        epochs     = 150,
        batch_size = 32
    )
)

Classificação do cubo de *embeddings*, gerando o cubo de probabilidades:

In [ ]:
emb_probs <- sits_classify(
    data       = cube_emb,
    ml_model   = mlp_model,
    memsize    = 4,
    multicores = 2,
    output_dir = output_dir,
    version    = "mlp"
)

Suavização espacial das probabilidades:

In [ ]:
emb_smooth <- sits_smooth(
    cube       = emb_probs,
    memsize    = 4,
    multicores = 2,
    output_dir = output_dir,
    version    = "mlp"
)

Geração do mapa final:

In [ ]:
emb_map <- sits_label_classification(
    cube       = emb_smooth,
    memsize    = 4,
    multicores = 2,
    output_dir = output_dir,
    version    = "mlp"
)

In [ ]:
plot(emb_map)

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Para aprofundar, consulte os capítulos <a href="https://e-sensing.github.io/sitsbook/emb_methods.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Embedding Methods in SITS</a>, <a href="https://e-sensing.github.io/sitsbook/emb_build.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Building regional embeddings: a simple example</a> e <a href="https://e-sensing.github.io/sitsbook/emb_example.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Building regional embeddings: a full example</a> do livro oficial do <code>sits</code>.
</div>